# 🧹 Preprocesamiento — Detección de toxicidad en YouTube (`IsToxic`)

## 🎯 Objetivo
Pipeline completo de limpieza, lematización, split y data augmentation sobre
`youtoxic_english_1000.csv` con variable objetivo `IsToxic`.

## ⚠️ Decisión clave: split ANTES del augmentation
El augmentation se aplica **únicamente sobre el conjunto de entrenamiento** tras el split.
Si se hace antes, versiones sintéticas de textos del train pueden acabar en test →
métricas infladas → evaluación inválida en producción.

## 📋 Flujo del notebook
1. Setup e imports
2. Carga y selección de columnas
3. Limpieza con regex
4. Lematización con spaCy
5. Split estratificado train / val / test
6. Data Augmentation solo sobre train
7. Exportación de CSVs a `data/processed/V_04`

In [3]:
import re
import os
import time
import warnings

import pandas as pd
import numpy as np
import nltk
import spacy
from tqdm import tqdm
from deep_translator import GoogleTranslator
import nlpaug.augmenter.word as naw
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Recursos NLTK necesarios
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet',
                 'omw-1.4', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

# Crear carpeta de salida si no existe
os.makedirs('../data/processed/V_04', exist_ok=True)

TARGET       = 'IsToxic'
RANDOM_STATE = 42
AUG_THRESHOLD = 0.48  # ratio mínimo de clase tóxica en train antes de augmentar

print('Setup completado.')
print(f'Carpeta de salida: ../data/processed/V_04')

Setup completado.
Carpeta de salida: ../data/processed/V_04


## 📂 1. Carga del dataset y selección de columnas

Se cargan solo las columnas necesarias:
- `CommentId` y `VideoId` para trazabilidad
- `Text` como entrada de texto
- `IsToxic` como variable objetivo

In [5]:
df_raw = pd.read_csv('../../data/raw/youtoxic_english_1000.csv')

print('Shape completo:', df_raw.shape)
print('Columnas      :', df_raw.columns.tolist())
print('\nNulos:\n', df_raw.isnull().sum())

df = df_raw[['CommentId', 'VideoId', 'Text', TARGET]].copy()

print('\nShape tras selección de columnas:', df.shape)
print('\nDistribución de IsToxic:')
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(3))

Shape completo: (1000, 15)
Columnas      : ['CommentId', 'VideoId', 'Text', 'IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']

Nulos:
 CommentId          0
VideoId            0
Text               0
IsToxic            0
IsAbusive          0
IsThreat           0
IsProvocative      0
IsObscene          0
IsHatespeech       0
IsRacist           0
IsNationalist      0
IsSexist           0
IsHomophobic       0
IsReligiousHate    0
IsRadicalism       0
dtype: int64

Shape tras selección de columnas: (1000, 4)

Distribución de IsToxic:
IsToxic
False    538
True     462
Name: count, dtype: int64
IsToxic
False    0.538
True     0.462
Name: proportion, dtype: float64


## 🧹 2. Limpieza de texto con expresiones regulares

Pipeline aplicado en orden:
1. Minúsculas
2. Eliminación de URLs (`http`, `www`, `https`)
3. Eliminación de menciones (`@usuario`) y hashtags (`#tag`)
4. Eliminación de dígitos
5. Solo letras y espacios
6. Normalización de espacios múltiples

Al final se detectan y eliminan filas cuyo texto quedó vacío tras la limpieza.

In [12]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Ejemplo paso a paso
ejemplo = df['Text'].iloc[1]
print('Original :', ejemplo)
print('Limpio   :', clean_text(ejemplo))

df['clean_text'] = df['Text'].apply(clean_text)

# Detectar y eliminar textos vacíos
vacios = (df['clean_text'] == '').sum()
print(f'\nTextos vacíos tras limpieza: {vacios}')
if vacios > 0:
    df = df[df['clean_text'] != ''].reset_index(drop=True)
    print(f'Shape tras eliminar vacíos: {df.shape}')

# Estadísticas de reducción
antes   = df['Text'].str.len().mean()
despues = df['clean_text'].str.len().mean()
print(f'\nLongitud media antes  : {antes:.0f} caracteres')
print(f'Longitud media después: {despues:.0f} caracteres')
print(f'Reducción             : {(1 - despues/antes)*100:.1f}%')

df[['Text', 'clean_text']].head(3)

Original : Law enforcement is not trained to shoot to apprehend.  They are trained to shoot to kill.  And I thank Wilson for killing that punk bitch.
Limpio   : law enforcement is not trained to shoot to apprehend they are trained to shoot to kill and i thank wilson for killing that punk bitch

Textos vacíos tras limpieza: 0

Longitud media antes  : 186 caracteres
Longitud media después: 177 caracteres
Reducción             : 4.8%


,Text,clean_text
0,If only people would just take a step back and...,if only people would just take a step back and...
1,Law enforcement is not trained to shoot to app...,law enforcement is not trained to shoot to app...
2,\r\nDont you reckon them 'black lives matter' ...,dont you reckon them black lives matter banner...


## 🧠 3. Lematización con spaCy

Se usa el modelo `en_core_web_sm` de **spaCy**, que aplica lematización con contexto
morfológico completo (POS tagging interno). Esto es más preciso que `WordNetLemmatizer`
de NLTK, que sin POS tag lematiza incorrectamente verbos y adjetivos comparativos.

Por cada token se eliminan:
- **Stopwords**
- **Signos de puntuación**
- **Lemas vacíos** resultantes de la limpieza previa

Al final se filtran las filas cuyo `lemmatized_text` quedó vacío.

In [13]:
nlp = spacy.load('en_core_web_sm')

docs = nlp.pipe(df['clean_text'].tolist(), batch_size=50, n_process=1)

df['text_procesado'] = [
    ' '.join([
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct and token.lemma_.strip()
    ])
    for doc in tqdm(docs, total=len(df), desc='Lematizando')
]

# Eliminar columna intermedia
df = df.drop(columns=['clean_text'])

# Filtrar vacíos
vacios_lemma = (df['text_procesado'].str.strip() == '').sum()
print(f'Textos vacíos tras lematización: {vacios_lemma}')
if vacios_lemma > 0:
    df = df[df['text_procesado'].str.strip() != ''].reset_index(drop=True)
    print(f'Shape tras filtrar: {df.shape}')

# Verificar columnas finales
print('\nColumnas del DataFrame:', df.columns.tolist())

# Guardar CSV preprocesado completo
df.to_csv('../../data/processed/V_04/youtoxic_preprocessed_IsToxic.csv', index=False)
print('CSV preprocesado guardado en data/processed/V_04/')

df[['Text', 'text_procesado', TARGET]].head(5)

Lematizando: 100%|██████████| 997/997 [00:06<00:00, 151.62it/s]

Textos vacíos tras lematización: 0

Columnas del DataFrame: ['CommentId', 'VideoId', 'Text', 'IsToxic', 'text_procesado']
CSV preprocesado guardado en data/processed/V_04/


,Text,text_procesado,IsToxic
0,If only people would just take a step back and...,people step case not people situation lump mes...,False
1,Law enforcement is not trained to shoot to app...,law enforcement train shoot apprehend train sh...,True
2,\r\nDont you reckon them 'black lives matter' ...,not reckon black life matter banner hold white...,True
3,There are a very large number of people who do...,large number people like police officer call c...,False
4,"The Arab dude is absolutely right, he should h...",arab dude absolutely right shoot extra time sh...,False


## ✂️ 4. Split estratificado — Train / Validation / Test

Split **antes** de cualquier augmentation para evitar data leakage.

| Conjunto | Proporción | Filas aprox. | Uso |
|---|---|---|---|
| **Train** | 70 % | ~700 | Entrenar el modelo + augmentation |
| **Validation** | 15 % | ~150 | Ajuste de hiperparámetros (Optuna + MLflow) |
| **Test** | 15 % | ~150 | Evaluación final — **no se toca** |

`stratify=y` garantiza que la proporción de `IsToxic` se mantenga
igual en los tres conjuntos.

In [14]:
X = df['text_procesado']
y = df[TARGET].astype(int)

# Paso 1: separar train (70%) del resto (30%)
X_temp, X_test_raw, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE
)

# Paso 2: separar val (15%) y test (15%) del resto
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15 / 0.85,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print('RESULTADO DEL SPLIT:')
print(f'  Train : {len(X_train_raw):>4} filas  |  tóxico: {y_train.mean():.1%}')
print(f'  Val   : {len(X_val_raw):>4} filas  |  tóxico: {y_val.mean():.1%}')
print(f'  Test  : {len(X_test_raw):>4} filas  |  tóxico: {y_test.mean():.1%}')

# Recuperar filas completas por índice para guardar con todas las columnas
df_val  = df.loc[X_val_raw.index].copy()
df_test = df.loc[X_test_raw.index].copy()

df_val.to_csv('../../data/processed/V_04/val.csv', index=False)
df_test.to_csv('../../data/processed/V_04/test.csv', index=False)
print('\nVal y Test guardados — no se volverán a tocar.')

RESULTADO DEL SPLIT:
  Train :  697 filas  |  tóxico: 46.3%
  Val   :  150 filas  |  tóxico: 46.0%
  Test  :  150 filas  |  tóxico: 46.0%

Val y Test guardados — no se volverán a tocar.


## 🧪 5. Data Augmentation — solo sobre Train

### Estrategia diferenciada por clase

| Clase | Técnica | Motivo |
|---|---|---|
| **Tóxicos** | Solo Back-Translation (🇩🇪 alemán) | Synonym replacement puede cambiar palabras clave y el texto deja de ser tóxico |
| **Normales** | Back-Translation (🇩🇪 alemán) + Synonym Replacement | Ambas técnicas son seguras en textos no tóxicos |

### Idioma intermedio: Alemán
El alemán tiene estructura gramatical suficientemente distinta al inglés
(orden de palabras SOV, compuestos nominales) para introducir variación
léxica real sin perder el significado del texto.

### Objetivo de balance
No se fuerza un 50/50 artificial. Con un balance inicial ~46/54 en train,
se aumenta solo hasta alcanzar `AUG_THRESHOLD = 0.48`, generando el mínimo
de muestras sintéticas necesario para reducir el desbalance sin introducir ruido.

### Validación de calidad
Si el texto aumentado es idéntico al original (la API devolvió el mismo texto),
se conserva el original pero no se cuenta como muestra nueva.

In [15]:
def back_translation(text, lang='de', max_chars=300, retries=3):
    """
    Traduce text inglés → lang → inglés.
    Retry con backoff exponencial si la API falla.
    Devuelve el texto original si todos los intentos fallan.
    """
    text = text[:max_chars]
    for attempt in range(retries):
        try:
            intermedio = GoogleTranslator(source='en', target=lang).translate(text)
            time.sleep(0.4)
            resultado  = GoogleTranslator(source=lang, target='en').translate(intermedio)
            time.sleep(0.4)
            # Si el resultado es idéntico al original, devolver original
            if resultado and resultado.strip().lower() != text.strip().lower():
                return resultado
            return text
        except Exception:
            time.sleep(2 ** attempt)  # backoff: 1s, 2s, 4s
    return text


aug_synonym = naw.SynonymAug(aug_src='wordnet', aug_p=0.3)

def safe_synonym(text):
    """Aplica synonym replacement con fallback al original."""
    try:
        result = aug_synonym.augment(text)
        return result[0] if result else text
    except Exception:
        return text


# Prueba rápida de las funciones
print('Probando back-translation...')
ejemplo = 'You are a stupid idiot and I hate you'
resultado = back_translation(ejemplo)
print(f'  Original : {ejemplo}')
print(f'  Resultado: {resultado}')

print('\nProbando synonym replacement...')
ejemplo_n = 'This video is really interesting and informative'
resultado_n = safe_synonym(ejemplo_n)
print(f'  Original : {ejemplo_n}')
print(f'  Resultado: {resultado_n}')

Probando back-translation...
  Original : You are a stupid idiot and I hate you
  Resultado: You're a stupid idiot and I hate you

Probando synonym replacement...
  Original : This video is really interesting and informative
  Resultado: This tv is really interesting and informative


In [17]:
train_df = df.loc[X_train_raw.index].copy().reset_index(drop=True)

minority_ratio = train_df[TARGET].mean()
print(f'Ratio tóxico en train antes del aug: {minority_ratio:.2%}')
print(f'Umbral objetivo                    : {AUG_THRESHOLD:.2%}')

nuevas_filas = []

if minority_ratio < AUG_THRESHOLD:

    n_toxic  = (train_df[TARGET] == True).sum()
    n_normal = (train_df[TARGET] == False).sum()
    n_needed = int(n_normal * AUG_THRESHOLD / (1 - AUG_THRESHOLD)) - n_toxic
    n_needed = min(n_needed, n_toxic)

    # ── Tóxicos: solo back-translation ────────────────────────────────────────
    print(f'\nGenerando {n_needed} muestras tóxicas con back-translation (🇩🇪 alemán)...')
    textos_toxicos = (train_df[train_df[TARGET] == True]['text_procesado']
                      .sample(n=n_needed, random_state=RANDOM_STATE).values)

    for i, texto in enumerate(tqdm(textos_toxicos, desc='BT tóxicos')):
        aug_text = back_translation(texto, lang='de')
        nuevas_filas.append({
            'CommentId':      f'AUG_TOX_{i+1}',
            'VideoId':        'augmented',
            'Text':           aug_text,
            'text_procesado': aug_text,
            TARGET:           True
        })

    # ── Normales: back-translation + synonym replacement ──────────────────────
    n_normal_needed = n_needed // 2

    textos_normales = (train_df[train_df[TARGET] == False]['text_procesado']
                       .sample(n=n_normal_needed * 2, random_state=RANDOM_STATE).values)

    print(f'\nGenerando {n_normal_needed} normales con back-translation...')
    for i, texto in enumerate(tqdm(textos_normales[:n_normal_needed], desc='BT normales')):
        aug_text = back_translation(texto, lang='de')
        nuevas_filas.append({
            'CommentId':      f'AUG_NOR_BT_{i+1}',
            'VideoId':        'augmented',
            'Text':           aug_text,
            'text_procesado': aug_text,
            TARGET:           False
        })

    print(f'\nGenerando {n_normal_needed} normales con synonym replacement...')
    for i, texto in enumerate(tqdm(textos_normales[n_normal_needed:], desc='SR normales')):
        aug_text = safe_synonym(texto)
        nuevas_filas.append({
            'CommentId':      f'AUG_NOR_SR_{i+1}',
            'VideoId':        'augmented',
            'Text':           aug_text,
            'text_procesado': aug_text,
            TARGET:           False
        })

    aug_df   = pd.DataFrame(nuevas_filas)
    train_df = pd.concat([train_df, aug_df], ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    print(f'\nTrain tras augmentation: {len(train_df)} filas')
    print(train_df[TARGET].value_counts())
    print(train_df[TARGET].value_counts(normalize=True).round(3))

else:
    print('Augmentation omitido: el balance en train ya es aceptable.')

Ratio tóxico en train antes del aug: 46.34%
Umbral objetivo                    : 48.00%

Generando 22 muestras tóxicas con back-translation (🇩🇪 alemán)...


BT tóxicos: 100%|██████████| 22/22 [00:46<00:00,  2.09s/it]



Generando 11 normales con back-translation...


BT normales: 100%|██████████| 11/11 [00:21<00:00,  1.95s/it]



Generando 11 normales con synonym replacement...


SR normales: 100%|██████████| 11/11 [00:00<00:00, 559.66it/s]


Train tras augmentation: 741 filas
IsToxic
False    396
True     345
Name: count, dtype: int64
IsToxic
False    0.534
True     0.466
Name: proportion, dtype: float64
